<a href="https://colab.research.google.com/github/fawaztanigbola/Nigeria-ForeCasting-Model/blob/main/Combine_Yearly_Files.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

from pathlib import Path
import xarray as xr

from google.colab import drive
drive.mount('/content/drive')


daily_folder = Path(
    "/content/drive/MyDrive/Nigeria_Temperature_Project/processed"
)

# Find only the yearly daily-temperature files
files = sorted(daily_folder.glob("daily_temperature_*.nc"))

# Confirm that all ten years are available
print("Files found:", len(files))
print([f.name for f in files])

expected_years = set(range(2015, 2025))
found_years = {int(f.stem.split("_")[-1]) for f in files}

missing_years = expected_years - found_years

if missing_years:
    raise ValueError(f"Missing yearly files: {sorted(missing_years)}")

# Combine yearly files while keeping elevation spatially static
daily_ds = xr.open_mfdataset(
    files,
    combine="by_coords",
    data_vars="minimal",
    coords="minimal",
    compat="override",
    chunks={"valid_time": 30}
)

print(daily_ds)


Mounted at /content/drive
Files found: 10
['daily_temperature_2015.nc', 'daily_temperature_2016.nc', 'daily_temperature_2017.nc', 'daily_temperature_2018.nc', 'daily_temperature_2019.nc', 'daily_temperature_2020.nc', 'daily_temperature_2021.nc', 'daily_temperature_2022.nc', 'daily_temperature_2023.nc', 'daily_temperature_2024.nc']
<xarray.Dataset> Size: 387MB
Dimensions:     (valid_time: 3653, latitude: 101, longitude: 131)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 29kB 2015-01-01 ... 2024-12-31
  * latitude    (latitude) float64 808B 14.0 13.9 13.8 13.7 ... 4.3 4.2 4.1 4.0
  * longitude   (longitude) float64 1kB 2.0 2.1 2.2 2.3 ... 14.7 14.8 14.9 15.0
    number      int64 8B ...
Data variables:
    tmin        (valid_time, latitude, longitude) float32 193MB dask.array<chunksize=(30, 101, 131), meta=np.ndarray>
    tmax        (valid_time, latitude, longitude) float32 193MB dask.array<chunksize=(30, 101, 131), meta=np.ndarray>
    elevation   (latitude, longitude) float

Save Combined Dataset.

In [3]:

output_file = daily_folder / "combined_daily_temperature_2015_2024.nc"

daily_ds.to_netcdf(output_file)

print("Combined dataset saved:", output_file)


Combined dataset saved: /content/drive/MyDrive/Nigeria_Temperature_Project/processed/combined_daily_temperature_2015_2024.nc


Confirmation of dataset

In [4]:

print("First date:", daily_ds.valid_time.values[0])
print("Last date:", daily_ds.valid_time.values[-1])
print("Total days:", daily_ds.sizes["valid_time"])

print("Temperature dimensions:", daily_ds["tmin"].dims)
print("Elevation dimensions:", daily_ds["elevation"].dims)


First date: 2015-01-01T00:00:00.000000000
Last date: 2024-12-31T00:00:00.000000000
Total days: 3653
Temperature dimensions: ('valid_time', 'latitude', 'longitude')
Elevation dimensions: ('latitude', 'longitude')


In [5]:
print(daily_ds.values)

<bound method Mapping.values of <xarray.Dataset> Size: 387MB
Dimensions:     (valid_time: 3653, latitude: 101, longitude: 131)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 29kB 2015-01-01 ... 2024-12-31
  * latitude    (latitude) float64 808B 14.0 13.9 13.8 13.7 ... 4.3 4.2 4.1 4.0
  * longitude   (longitude) float64 1kB 2.0 2.1 2.2 2.3 ... 14.7 14.8 14.9 15.0
    number      int64 8B ...
Data variables:
    tmin        (valid_time, latitude, longitude) float32 193MB dask.array<chunksize=(30, 101, 131), meta=np.ndarray>
    tmax        (valid_time, latitude, longitude) float32 193MB dask.array<chunksize=(30, 101, 131), meta=np.ndarray>
    elevation   (latitude, longitude) float32 53kB dask.array<chunksize=(101, 131), meta=np.ndarray>>
